# Main-equivalent `build_dictionary` · LeiData wav1 · windows 1–2

Views the last one-shot build (main-branch knobs, `--n-windows 2`):

| Knob | Value |
|---|---|
| `--n-windows` / offset | 2 / 0 |
| `--n-rounds` | 4 |
| `--eps-step-fraction` | 0.2 |
| `--min-repeats` | 3 |
| `--max-template-width` | 24 |
| `--refine-iters` | 2 |
| `--template-method` | `average` |

Catalog: `dictionary_id=66` · out dir under `wav1_averaged_w2_mainlike/`.

In [ ]:
%matplotlib inline

import json
import sys
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

REPO = Path("/home/yfreund/projects/gplrev3-webserver")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

LIBRARY_ROOT = Path("/home/yfreund/whale_processing")
OUT_ROOT = LIBRARY_ROOT / "result/LeiData/dict_learning_templates/wav1_averaged_w2_mainlike"
DB_PATH = LIBRARY_ROOT / "result/dict_learning_catalog.sqlite"
WAV = LIBRARY_ROOT / "LeiData/SOCAL_H_76_230427_003345.x.wav"
DICTIONARY_ID = 66


def latest_run_dir(root: Path) -> Path:
    stamped = sorted(
        [p for p in root.iterdir() if p.is_dir() and (p / "manifest.json").is_file()],
        key=lambda p: p.name,
    )
    if not stamped:
        raise FileNotFoundError(f"No stamped run under {root}")
    return stamped[-1]


OUT_DIR = latest_run_dir(OUT_ROOT)
man = json.loads((OUT_DIR / "manifest.json").read_text())
params = man.get("params") or {}
e0 = man.get("energy_before")
e1 = man.get("energy_after")
frac = None
if e0 is not None and e1 is not None and float(e0) > 0:
    frac = (float(e0) - float(e1)) / float(e0)

print(f"OUT_DIR: {OUT_DIR}")
print(
    f"templates={len(man.get('templates') or [])}  "
    f"matchings={len(man.get('matchings') or [])}  "
    f"stop={man.get('stop_reason')}  "
    f"%E removed={None if frac is None else f'{100*frac:.1f}%'}"
)

## Run summary

In [ ]:
pct = "—" if frac is None else f"{100 * frac:.1f}%"
lines = [
    "| Field | Value |",
    "|---|---|",
    f"| `dictionary_id` | **{DICTIONARY_ID}** |",
    f"| name | `{man.get('name')}` |",
    f"| `n_windows` | {params.get('n_windows')} |",
    f"| `n_rounds` | {params.get('n_rounds')} |",
    f"| `eps_step_fraction` | {params.get('eps_step_fraction')} |",
    f"| `min_repeats` | {params.get('min_repeats')} |",
    f"| `max_template_width` | {params.get('max_template_width')} |",
    f"| `refine_iters` | {params.get('refine_iters')} |",
    f"| `template_method` | `{params.get('template_method')}` |",
    f"| templates / matchings | {len(man.get('templates') or [])} / {len(man.get('matchings') or [])} |",
    f"| %E removed | {pct} |",
    f"| `stop_reason` | `{man.get('stop_reason')}` |",
    f"| `out_dir` | `{OUT_DIR}` |",
]
display(Markdown("\n".join(lines)))

print("window_ids:")
for wid in man.get("window_ids") or []:
    print(" ", wid)

## Outer-round energy log

In [ ]:
logs = man.get("round_logs") or []
if not logs:
    print("No round_logs in manifest.")
else:
    rows = [
        "| round | seed matches | avg matches | ΔE | cum ΔE | time coverage |",
        "|---:|---:|---:|---:|---:|---:|",
    ]
    rounds, dEs, covs = [], [], []
    for i, row in enumerate(logs):
        r = row if isinstance(row, dict) else {}
        rnd = r.get("round_index", r.get("round", i))
        seed_n = r.get("seed_n_matchings", r.get("seed_matches"))
        avg_n = r.get("averaged_n_matchings", r.get("avg_matches"))
        dE = r.get("delta_energy", r.get("dE"))
        cum = r.get("cumulative_delta_energy", r.get("cum_dE"))
        cov = r.get("time_coverage_fraction", r.get("time_bin_coverage"))
        if isinstance(cov, dict):
            cov = cov.get("fraction")
        cov_s = "—" if cov is None else f"{100 * float(cov):.1f}%"
        dE_s = "—" if dE is None else f"{float(dE):.3e}"
        cum_s = "—" if cum is None else f"{float(cum):.3e}"
        rows.append(
            f"| {rnd} | {seed_n} | {avg_n} | {dE_s} | {cum_s} | {cov_s} |"
        )
        rounds.append(rnd)
        if dE is not None:
            dEs.append(float(dE))
        if cov is not None:
            covs.append(float(cov))
    display(Markdown("\n".join(rows)))

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
    if dEs:
        axes[0].bar([str(r) for r in rounds[: len(dEs)]], dEs, color="steelblue")
        axes[0].set_xlabel("round"); axes[0].set_ylabel("ΔE")
        axes[0].set_title("Energy removed per outer round")
    if covs:
        axes[1].plot(rounds[: len(covs)], covs, "o-")
        axes[1].set_xlabel("round"); axes[1].set_ylabel("time-bin coverage")
        axes[1].set_ylim(0, 1)
        axes[1].set_title("Coverage after each round")
    fig.tight_layout()
    plt.show()

## Learned templates

In [ ]:
templates = []
for item in man.get("templates") or []:
    arr = np.load(OUT_DIR / item["file"])
    view = arr[0] if arr.ndim == 3 else arr
    templates.append((int(item["template_id"]), view, item))

n = len(templates)
ncols = min(4, max(n, 1))
nrows = int(np.ceil(n / ncols)) if n else 1
fig, axes = plt.subplots(nrows, ncols, figsize=(3.0 * ncols, 2.8 * nrows), squeeze=False)
for ax in axes.ravel():
    ax.axis("off")
for i, (tid, view, meta) in enumerate(templates):
    ax = axes[i // ncols][i % ncols]
    # Full min–max is dominated by a few large spikes → most pixels map to black.
    # Use robust percentiles so near-zero structure is visible.
    vmin, vmax = np.percentile(view, [1, 99])
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin >= vmax:
        vmin, vmax = float(np.min(view)), float(np.max(view))
    im = ax.imshow(view, aspect="auto", origin="lower", cmap="magma", vmin=vmin, vmax=vmax)
    w = meta.get("width") or view.shape[-1]
    ax.set_title(
        f"id={tid}  {view.shape[0]}×{w}\n"
        f"[{view.min():.2g}, {view.max():.2g}] clim p1–p99",
        fontsize=9,
    )
    ax.set_xlabel("time"); ax.set_ylabel("freq")
    ax.axis("on")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle(f"Templates · dict {DICTIONARY_ID} · average · refine_iters=2", y=1.02)
fig.tight_layout()
plt.show()

print("Template stats (not constant — heavy right tail):")
for tid, view, _ in templates:
    print(
        f"  tid={tid}: min={view.min():.4g} max={view.max():.4g} "
        f"mean={view.mean():.4g} std={view.std():.4g} "
        f"p50={np.percentile(view,50):.4g} p99={np.percentile(view,99):.4g}"
    )

counts = Counter(int(m["template_id"]) for m in man.get("matchings") or [])
if counts:
    tids = sorted(counts)
    fig, ax = plt.subplots(figsize=(max(4, 0.55 * len(tids)), 3))
    ax.bar([str(t) for t in tids], [counts[t] for t in tids], color="steelblue")
    ax.set_xlabel("template_id"); ax.set_ylabel("# matchings")
    ax.set_title("Matchings per atom (after refine)")
    for tid in tids:
        ax.text(str(tid), counts[tid], str(counts[tid]), ha="center", va="bottom")
    fig.tight_layout()
    plt.show()

## Spectrogram overlays (windows 1 and 2)

Load both whitened windows from the catalog and draw matching intervals.
Spectrogram uses `jet` with 1st–99th percentile color limits (same spike issue as templates).

In [ ]:
from database import catalog


def load_windows(db_path: Path, wav: Path, n: int = 2) -> list[tuple[str, np.ndarray]]:
    with catalog.connect(db_path) as conn:
        wav_row = catalog.find_wav_file(conn, str(wav))
        if wav_row is None:
            raise FileNotFoundError(wav)
        segs = catalog.list_gpl_segmentations_for_wav(conn, wav_row["wav_id"])
        wins = catalog.list_windows_for_segmentation(
            conn, segs[0]["segmentation_id"], limit=n, offset=0
        )
        return [(w["window_id"], np.load(w["npy_path"])) for w in wins]


def template_widths(man: dict) -> dict[int, int]:
    widths = {}
    for t in man.get("templates") or []:
        tid = int(t["template_id"])
        if t.get("width") is not None:
            widths[tid] = int(t["width"])
        else:
            shape = t.get("shape") or []
            widths[tid] = int(shape[-1]) if shape else 1
    return widths


windows = load_windows(DB_PATH, WAV, n=2)
widths = template_widths(man)
by_window = defaultdict(list)
for m in man.get("matchings") or []:
    by_window[m["window_id"]].append(m)
cmap = plt.cm.tab10

fig, axes = plt.subplots(len(windows), 1, figsize=(14, 3.4 * len(windows)), sharex=False)
if len(windows) == 1:
    axes = [axes]

for ax, (wid, spec) in zip(axes, windows):
    vmin, vmax = np.percentile(spec, [1, 99])
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin >= vmax:
        vmin, vmax = float(np.min(spec)), float(np.max(spec))
    ax.imshow(spec, aspect="auto", origin="lower", cmap="jet", vmin=vmin, vmax=vmax)
    drawn = 0
    for m in by_window.get(wid, []):
        tid = int(m["template_id"])
        t0 = int(m["t_start"])
        w = widths.get(tid, 1)
        ax.axvspan(t0, t0 + w, color="white", alpha=0.25)
        ax.axvline(t0, color=cmap(tid % 10), lw=1.2, alpha=0.95)
        ax.axvline(t0 + w, color=cmap(tid % 10), lw=1.0, alpha=0.7, ls="--")
        drawn += 1
    ax.set_title(
        f"{wid[-40:]} · {drawn} matchings · clim p1–p99 "
        f"[{vmin:.3g}, {vmax:.3g}]"
    )
    ax.set_ylabel("freq")
axes[-1].set_xlabel("time bin")
fig.suptitle(f"dict {DICTIONARY_ID} matching overlays (jet)", y=1.01)
fig.tight_layout()
plt.show()

print("Matchings by window:")
for wid, _ in windows:
    ms = sorted(by_window.get(wid, []), key=lambda x: (x["t_start"], x["template_id"]))
    print(f"  {wid}: {len(ms)}")
    for m in ms:
        tid = int(m["template_id"])
        t0 = int(m["t_start"])
        w = widths.get(tid, 1)
        amp = m.get("amplitude")
        amp_s = f"{float(amp):.3f}" if amp is not None else "—"
        print(f"    tid={tid}  t=[{t0}, {t0 + w})  amp={amp_s}")

## Catalog cross-check

In [ ]:
with catalog.connect(DB_PATH) as conn:
    d = catalog.get_dictionary(conn, DICTIONARY_ID)
    tmpls = catalog.list_dictionary_templates(conn, DICTIONARY_ID)
    matches = catalog.list_matchings(conn, DICTIONARY_ID)
    wins = catalog.list_dictionary_windows(conn, DICTIONARY_ID)

print(
    f"dictionary_id={DICTIONARY_ID}  name={d.get('name')!r}  source={d.get('source')}  "
    f"parent={d.get('parent_dictionary_id')}"
)
print(f"catalog: templates={len(tmpls)}  windows={len(wins)}  matchings={len(matches)}")
assert len(tmpls) == len(man.get("templates") or [])
assert len(matches) == len(man.get("matchings") or [])
print("OK — catalog agrees with manifest counts.")